# 16: Spark & Sedona Distributed Operations

This notebook demonstrates the **distributed computing** capabilities of siege_utilities,
covering PySpark, Apache Sedona, and HDFS operations.

## Key Features

1. **HDFSConfig & Session Factories** — Preconfigured Spark sessions for different workloads
2. **DataFrame Operations (dual-mode)** — Pandas first, Spark equivalent alongside
3. **I/O Operations** — Parquet, CSV, Excel, atomic writes
4. **JSON Flattening** — Shallow and deep modes for nested JSON columns
5. **Pivot & Summary Tables** — Boolean flag summaries and categorical pivots
6. **Sedona Spatial Operations** — Geometry validation, reprojection
7. **HDFS Operations** — Status checks, sync, distributed environment setup
8. **Walkability Helper** — Distance-based walkability grading

## Dual-Mode Pattern

Every data section shows **pandas first**, then the equivalent Spark operation gated by
`SPARK_AVAILABLE`. Both paths produce equivalent results, so this notebook runs correctly
whether or not PySpark is installed.

## Prerequisites

```bash
pip install -e /path/to/siege_utilities[distributed]  # PySpark + Sedona
# OR: run in pandas-only mode (no extra install needed)
```

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd().parent))

import siege_utilities as su
su.configure_shared_logging(level="INFO")

import pandas as pd
import numpy as np

su.log_info("Base imports successful")

# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_16"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

su.log_info(f"Output directory: {OUTPUT_DIR}  (exists={OUTPUT_DIR.exists()})")

## 1. Spark Availability Detection

This cell establishes the **dual-mode pattern** used throughout all data notebooks.
PySpark and Sedona are detected at import time; all subsequent cells gracefully
degrade to pandas-only when Spark is unavailable.

In [ ]:
# --- Spark & Sedona availability detection ---
SPARK_AVAILABLE = False
SEDONA_AVAILABLE = False
spark = None

try:
    from pyspark.sql import SparkSession, DataFrame as SparkDataFrame
    from pyspark.sql import functions as F
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
    SPARK_AVAILABLE = True
    su.log_info("PySpark is available")
except ImportError:
    su.log_info("PySpark not installed -- running in pandas-only mode")

if SPARK_AVAILABLE:
    try:
        from sedona.register import SedonaRegistrator
        SEDONA_AVAILABLE = True
        su.log_info("Apache Sedona is available")
    except ImportError:
        su.log_info("Sedona not installed -- spatial operations will be skipped")

su.log_info(f"SPARK_AVAILABLE = {SPARK_AVAILABLE}")
su.log_info(f"SEDONA_AVAILABLE = {SEDONA_AVAILABLE}")

## 2. HDFSConfig & Session Factories

The `HDFSConfig` dataclass is the central configuration object for all distributed operations.
Six factory functions create pre-tuned configurations for common workloads.

In [ ]:
from siege_utilities.distributed.hdfs_config import (
    HDFSConfig,
    create_hdfs_config,
    create_local_config,
    create_cluster_config,
    create_geocoding_config,
    create_yarn_config,
    create_census_analysis_config,
)

# Show all factory functions and their defaults
factories = {
    'create_local_config':           {'executors': 2, 'cores': 1, 'memory': '1g', 'sedona': False},
    'create_cluster_config':         {'executors': 8, 'cores': 4, 'memory': '4g', 'sedona': True},
    'create_geocoding_config':       {'executors': 4, 'cores': 2, 'memory': '2g', 'sedona': True},
    'create_yarn_config':            {'executors': 10, 'cores': 4, 'memory': '8g', 'sedona': True},
    'create_census_analysis_config': {'executors': 6, 'cores': 4, 'memory': '4g', 'sedona': True},
}

su.log_info("HDFSConfig Factory Functions")
su.log_info("=" * 70)
for name, defaults in factories.items():
    su.log_info(f"  {name:35s} executors={defaults['executors']}, "
                f"cores={defaults['cores']}, mem={defaults['memory']}, "
                f"sedona={defaults['sedona']}")

In [ ]:
# Create a local-mode config pointing at our OUTPUT_DIR
config = create_local_config(data_path=str(OUTPUT_DIR))

su.log_info(f"Config created:")
su.log_info(f"  app_name:       {config.app_name}")
su.log_info(f"  master:         {config.master}")
su.log_info(f"  is_local:       {config.is_local}")
su.log_info(f"  is_yarn:        {config.is_yarn}")
su.log_info(f"  num_executors:  {config.num_executors}")
su.log_info(f"  executor_cores: {config.executor_cores}")
su.log_info(f"  executor_memory:{config.executor_memory}")
su.log_info(f"  enable_sedona:  {config.enable_sedona}")
su.log_info(f"  cache_directory: {config.cache_directory}")
su.log_info(f"  optimal_partitions: {config.get_optimal_partitions()}")

In [ ]:
# Create a Spark session (if PySpark available)
if SPARK_AVAILABLE:
    from siege_utilities.distributed.hdfs_operations import AbstractHDFSOperations
    
    hdfs_ops = AbstractHDFSOperations(config)
    spark = hdfs_ops.create_spark_session()
    
    if spark:
        su.log_info(f"Spark session created: {spark.sparkContext.appName}")
        su.log_info(f"Spark version: {spark.version}")
        su.log_info(f"Master: {spark.sparkContext.master}")
    else:
        su.log_warning("Spark session creation failed -- falling back to pandas")
        SPARK_AVAILABLE = False
else:
    su.log_info("Skipping Spark session creation (PySpark not installed)")

## 3. DataFrame Operations (Dual-Mode)

Each operation is shown first in pandas, then in Spark. Both paths produce equivalent results.

In [ ]:
# Create sample data for dual-mode demos
np.random.seed(42)
sample_data = pd.DataFrame({
    'First Name': ['Alice', 'Bob', 'Carol', 'David', 'Eve'],
    'Last/Name': ['Smith', 'Jones', 'Williams', 'Brown', 'Davis'],
    'Age': [30, 25, 35, 28, 42],
    'Income ($)': [75000, 52000, 95000, 68000, 110000],
    'State': ['CA', 'TX', 'NY', 'CA', 'WA'],
})

su.log_info(f"Sample DataFrame: {sample_data.shape}")
display(sample_data)

In [ ]:
# --- 3a. Sanitize column names ---
# Pandas approach: lowercase, replace special chars with underscores
pdf_clean = sample_data.copy()
pdf_clean.columns = [c.lower().replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '').replace('$', '') for c in pdf_clean.columns]
su.log_info("Pandas -- sanitized columns:")
su.log_info(f"  {list(pdf_clean.columns)}")

# Spark approach
if SPARK_AVAILABLE and spark:
    from siege_utilities.distributed.spark_utils import sanitise_dataframe_column_names
    
    sdf = spark.createDataFrame(sample_data)
    sdf_clean = sanitise_dataframe_column_names(sdf)
    su.log_info("Spark -- sanitized columns:")
    su.log_info(f"  {sdf_clean.columns}")

In [ ]:
# --- 3b. Get row count ---
# Pandas
pdf_count = len(sample_data)
su.log_info(f"Pandas row count: {pdf_count}")

# Spark
if SPARK_AVAILABLE and spark:
    from siege_utilities.distributed.spark_utils import get_row_count
    sdf = spark.createDataFrame(sample_data)
    sdf_count = get_row_count(sdf)
    su.log_info(f"Spark row count: {sdf_count}")

In [ ]:
# --- 3c. Move column to front ---
# Pandas
cols = list(sample_data.columns)
cols.remove('State')
pdf_reordered = sample_data[['State'] + cols]
su.log_info(f"Pandas -- columns after move: {list(pdf_reordered.columns)}")

# Spark
if SPARK_AVAILABLE and spark:
    from siege_utilities.distributed.spark_utils import move_column_to_front_of_dataframe
    sdf = spark.createDataFrame(sample_data)
    sdf_reordered = move_column_to_front_of_dataframe(sdf, 'State')
    su.log_info(f"Spark -- columns after move: {sdf_reordered.columns}")

In [ ]:
# --- 3d. Repartition & Cache / Register Temp Table ---
if SPARK_AVAILABLE and spark:
    from siege_utilities.distributed.spark_utils import repartition_and_cache, register_temp_table
    
    sdf = spark.createDataFrame(sample_data)
    
    # Repartition and cache
    sdf_cached = repartition_and_cache(sdf, partitions=2)
    su.log_info(f"Repartitioned to {sdf_cached.rdd.getNumPartitions()} partitions and cached")
    
    # Register as temp table for SQL queries
    success = register_temp_table(sdf_cached, 'sample_people')
    su.log_info(f"Registered temp table 'sample_people': {success}")
    
    # Query it
    result = spark.sql("SELECT State, AVG(`Income ($)`) as avg_income FROM sample_people GROUP BY State")
    su.log_info("SQL query result:")
    result.show()
else:
    # Pandas equivalent
    result = sample_data.groupby('State')['Income ($)'].mean().reset_index()
    result.columns = ['State', 'avg_income']
    su.log_info("Pandas groupby result:")
    display(result)

## 4. I/O Operations

Read and write Parquet, CSV, and Excel files. Demonstrates atomic writes with staging directories.

In [ ]:
# --- 4a. Parquet I/O ---
parquet_path = OUTPUT_DIR / 'nb16_sample.parquet'

# Pandas write/read
sample_data.to_parquet(parquet_path, index=False)
pdf_loaded = pd.read_parquet(parquet_path)
su.log_info(f"Pandas: wrote & read {len(pdf_loaded)} rows from {parquet_path.name}")

# Spark write/read
if SPARK_AVAILABLE and spark:
    from siege_utilities.distributed.spark_utils import write_df_to_parquet, read_parquet_to_df
    
    sdf = spark.createDataFrame(sample_data)
    spark_parquet = str(OUTPUT_DIR / 'nb16_spark_sample.parquet')
    
    write_ok = write_df_to_parquet(sdf, spark_parquet)
    su.log_info(f"Spark: wrote parquet: {write_ok}")
    
    sdf_loaded = read_parquet_to_df(spark, spark_parquet)
    if sdf_loaded:
        su.log_info(f"Spark: read {get_row_count(sdf_loaded)} rows")

In [ ]:
# --- 4b. CSV Export with custom delimiter ---
csv_path = OUTPUT_DIR / 'nb16_sample.csv'

# Pandas
sample_data.to_csv(csv_path, sep='|', index=False)
su.log_info(f"Pandas: wrote pipe-delimited CSV to {csv_path.name}")

# Spark
if SPARK_AVAILABLE and spark:
    from siege_utilities.distributed.spark_utils import export_prepared_df_as_csv_to_path_using_delimiter
    
    sdf = spark.createDataFrame(sample_data)
    spark_csv_path = OUTPUT_DIR / 'nb16_spark_sample.csv'
    ok = export_prepared_df_as_csv_to_path_using_delimiter(sdf, spark_csv_path, delimiter='|')
    su.log_info(f"Spark: wrote pipe-delimited CSV: {ok}")

In [ ]:
# --- 4c. Excel Export ---
excel_path = OUTPUT_DIR / 'nb16_sample.xlsx'

# Pandas
sample_data.to_excel(excel_path, index=False, sheet_name='People')
su.log_info(f"Pandas: wrote Excel to {excel_path.name}")

# Spark (converts to Pandas internally)
if SPARK_AVAILABLE and spark:
    from siege_utilities.distributed.spark_utils import export_pyspark_df_to_excel
    
    sdf = spark.createDataFrame(sample_data)
    spark_excel = str(OUTPUT_DIR / 'nb16_spark_sample.xlsx')
    export_pyspark_df_to_excel(sdf, file_name=spark_excel, sheet_name='People')
    su.log_info(f"Spark: wrote Excel via toPandas()")

In [ ]:
# --- 4d. Atomic Write with Staging ---
if SPARK_AVAILABLE and spark:
    from siege_utilities.distributed.spark_utils import (
        atomic_write_with_staging, create_unique_staging_directory
    )
    
    sdf = spark.createDataFrame(sample_data)
    staging = create_unique_staging_directory(str(OUTPUT_DIR), 'nb16_atomic')
    final_dest = str(OUTPUT_DIR / 'nb16_atomic_output')
    
    ok = atomic_write_with_staging(sdf, final_dest, staging, file_format='csv', delimiter=',')
    su.log_info(f"Atomic write completed: {ok}")
    su.log_info(f"  Staging dir: {staging}")
    su.log_info(f"  Final dest:  {final_dest}")
else:
    su.log_info("Atomic write requires Spark -- skipped in pandas mode")

## 5. JSON Flattening

The `flatten_json_column_and_join_back_to_df()` function handles nested JSON columns
with both shallow and deep flattening modes, plus fallback for corrupt JSON.

In [ ]:
if SPARK_AVAILABLE and spark:
    from siege_utilities.distributed.spark_utils import flatten_json_column_and_join_back_to_df
    
    # Create sample data with a JSON column
    json_data = [
        (1, '{"name": "Alice", "address": {"city": "SF", "state": "CA"}}'),
        (2, '{"name": "Bob", "address": {"city": "Austin", "state": "TX"}}'),
        (3, '{"name": "Carol", "address": {"city": "NYC", "state": "NY"}}'),
    ]
    schema = StructType([
        StructField('id', IntegerType(), False),
        StructField('json_data', StringType(), True),
    ])
    sdf_json = spark.createDataFrame(json_data, schema)
    
    su.log_info("Before flattening:")
    sdf_json.show(truncate=False)
    
    # Shallow flatten
    sdf_flat = flatten_json_column_and_join_back_to_df(
        sdf_json, 'json_data', prefix='j_', flatten_level='shallow', verbose=True
    )
    su.log_info("After shallow flattening:")
    sdf_flat.show(truncate=False)
else:
    # Pandas equivalent for JSON flattening
    import json
    
    json_df = pd.DataFrame({
        'id': [1, 2, 3],
        'json_data': [
            '{"name": "Alice", "address": {"city": "SF", "state": "CA"}}',
            '{"name": "Bob", "address": {"city": "Austin", "state": "TX"}}',
            '{"name": "Carol", "address": {"city": "NYC", "state": "NY"}}',
        ]
    })
    
    parsed = json_df['json_data'].apply(json.loads)
    flat = pd.json_normalize(parsed)
    result = pd.concat([json_df[['id']], flat], axis=1)
    su.log_info("Pandas JSON normalization:")
    display(result)

## 6. Pivot & Summary Tables

Generate summary tables for boolean flags and categorical columns — useful for data quality reporting.

In [ ]:
# Create sample data with boolean flags
quality_data = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol', 'David', 'Eve', 'Frank', 'Grace', 'Henry'],
    'has_email': [True, True, False, True, True, False, True, True],
    'has_phone': [True, False, True, True, False, True, True, False],
    'is_verified': [True, True, True, False, True, False, True, True],
    'category': ['A', 'B', 'A', 'C', 'B', 'A', 'C', 'B'],
})

su.log_info("Quality data:")
display(quality_data)

In [ ]:
# --- 6a. Boolean pivot summary ---
# Pandas
bool_cols = ['has_email', 'has_phone', 'is_verified']
pdf_summary = pd.DataFrame({
    'field': bool_cols,
    'true_count': [quality_data[c].sum() for c in bool_cols],
    'false_count': [(~quality_data[c]).sum() for c in bool_cols],
    'true_pct': [quality_data[c].mean() * 100 for c in bool_cols],
})
su.log_info("Pandas -- Boolean summary:")
display(pdf_summary)

# Spark
if SPARK_AVAILABLE and spark:
    from siege_utilities.distributed.spark_utils import pivot_summary_table_for_bools
    
    sdf = spark.createDataFrame(quality_data)
    spark_summary = pivot_summary_table_for_bools(sdf, bool_cols, spark)
    su.log_info("Spark -- Boolean pivot summary:")
    spark_summary.show()

In [ ]:
# --- 6b. Categorical pivot with metrics ---
# Pandas
pdf_cat_summary = quality_data.groupby('category').agg(
    count=('name', 'count'),
    has_email_pct=('has_email', 'mean'),
    has_phone_pct=('has_phone', 'mean'),
).reset_index()
pdf_cat_summary['has_email_pct'] = (pdf_cat_summary['has_email_pct'] * 100).round(1)
pdf_cat_summary['has_phone_pct'] = (pdf_cat_summary['has_phone_pct'] * 100).round(1)
su.log_info("Pandas -- Category summary:")
display(pdf_cat_summary)

# Spark
if SPARK_AVAILABLE and spark:
    from siege_utilities.distributed.spark_utils import pivot_summary_with_metrics
    
    sdf = spark.createDataFrame(quality_data)
    spark_cat = pivot_summary_with_metrics(sdf, 'category', 'has_email', spark)
    su.log_info("Spark -- Category pivot:")
    spark_cat.show()

In [ ]:
# --- 6c. Prepare summary DataFrame ---
# Pandas
summary_tuples = [
    ('Total Records', str(len(quality_data))),
    ('Email Coverage', f"{quality_data['has_email'].mean():.0%}"),
    ('Phone Coverage', f"{quality_data['has_phone'].mean():.0%}"),
    ('Verification Rate', f"{quality_data['is_verified'].mean():.0%}"),
]
pdf_summary_df = pd.DataFrame(summary_tuples, columns=['metric', 'value'])
su.log_info("Pandas -- Summary DataFrame:")
display(pdf_summary_df)

# Spark
if SPARK_AVAILABLE and spark:
    from siege_utilities.distributed.spark_utils import prepare_summary_dataframe
    
    sdf_summary = prepare_summary_dataframe(summary_tuples)
    su.log_info("Spark -- Summary DataFrame:")
    sdf_summary.show()

## 7. Sedona Spatial Operations

These functions require Apache Sedona and are conditionally executed.

In [ ]:
if SPARK_AVAILABLE and SEDONA_AVAILABLE and spark:
    from siege_utilities.distributed.spark_utils import (
        validate_geometry,
        reproject_geom_columns,
        ensure_literal,
    )
    
    su.log_info("Sedona spatial functions available:")
    su.log_info("  validate_geometry(df, geom_col, step_name)")
    su.log_info("  reproject_geom_columns(df, geom_columns, source_srid, target_srid)")
    su.log_info("  ensure_literal(value) -- convert value to Spark Column")
    
    # Demonstrate ensure_literal
    from pyspark.sql import Column
    lit_val = ensure_literal(42)
    su.log_info(f"ensure_literal(42) -> {type(lit_val).__name__}")
    
    col_val = F.col('some_column')
    lit_col = ensure_literal(col_val)
    su.log_info(f"ensure_literal(Column) -> passes through (already a Column)")
else:
    su.log_info("Sedona not available -- spatial operations skipped")
    su.log_info("Install with: pip install apache-sedona")

## 8. HDFS Operations

The `AbstractHDFSOperations` class manages HDFS connectivity, directory sync, and
distributed environment setup.

In [ ]:
from siege_utilities.distributed.hdfs_operations import (
    AbstractHDFSOperations,
    setup_distributed_environment,
    create_hdfs_operations,
)

su.log_info("HDFS Operations API:")
su.log_info("  AbstractHDFSOperations(config)")
su.log_info("    .check_hdfs_status() -> bool")
su.log_info("    .create_spark_session() -> Optional[SparkSession]")
su.log_info("    .sync_directory_to_hdfs(local_path, hdfs_subdir) -> (path, info)")
su.log_info("    .setup_distributed_environment(data_path) -> (spark, path, None)")
su.log_info("")
su.log_info("  Convenience functions:")
su.log_info("    setup_distributed_environment(config, data_path)")
su.log_info("    create_hdfs_operations(config)")

In [ ]:
# Check HDFS status (will be False on local machines without Hadoop)
ops = create_hdfs_operations(config)
hdfs_ok = ops.check_hdfs_status()
su.log_info(f"HDFS accessible: {hdfs_ok}")

if not hdfs_ok:
    su.log_info("HDFS not available -- this is expected on local dev machines")
    su.log_info("The distributed environment will use local filesystem instead")

In [ ]:
# Full distributed environment setup (the recommended entry point)
if SPARK_AVAILABLE:
    env_spark, env_path, _ = setup_distributed_environment(config, str(OUTPUT_DIR))
    
    if env_spark:
        su.log_info(f"Distributed environment ready")
        su.log_info(f"  Data path: {env_path}")
        su.log_info(f"  Using: {'HDFS' if hdfs_ok else 'local filesystem'}")
    else:
        su.log_warning("Could not set up distributed environment")
else:
    su.log_info("Distributed environment requires PySpark -- skipped")

## 9. Walkability Helper

A simple utility that converts distance values to walkability grades.

In [ ]:
from siege_utilities.distributed.spark_utils import compute_walkability

# Test various distances (in meters)
test_distances = [100, 400, 800, 1200, 2000, 5000, None]

su.log_info("Walkability grades by distance:")
su.log_info(f"{'Distance (m)':>15}  {'Grade':>6}  {'Label'}")
su.log_info("-" * 45)

for d in test_distances:
    result = compute_walkability(d)
    if result:
        su.log_info(f"{str(d):>15}  {result['grade']:>6}  {result['label']}")
    else:
        su.log_info(f"{str(d):>15}  {'N/A':>6}  Invalid input")

## 10. Cleanup

In [ ]:
# Stop Spark session if running
if SPARK_AVAILABLE and spark:
    spark.stop()
    su.log_info("Spark session stopped")

# Clean up temp files
import shutil
for path in OUTPUT_DIR.glob('nb16_*'):
    if path.is_dir():
        shutil.rmtree(path)
    else:
        path.unlink()
su.log_info("Temp files cleaned up")

## Summary

### HDFSConfig Factory Functions

| Factory | Executors | Cores | Memory | Sedona | Use Case |
|---------|-----------|-------|--------|--------|----------|
| `create_local_config()` | 2 | 1 | 1g | No | Local development |
| `create_cluster_config()` | 8 | 4 | 4g | Yes | Production cluster |
| `create_geocoding_config()` | 4 | 2 | 2g | Yes | Geocoding pipelines |
| `create_yarn_config()` | 10 | 4 | 8g | Yes | YARN deployment |
| `create_census_analysis_config()` | 6 | 4 | 4g | Yes | Census longitudinal |

### Key Functions

| Module | Function | Purpose |
|--------|----------|----------|
| `hdfs_config` | `HDFSConfig` | Central configuration dataclass |
| `hdfs_operations` | `AbstractHDFSOperations` | HDFS + Spark session management |
| `spark_utils` | `sanitise_dataframe_column_names()` | Clean column names |
| `spark_utils` | `get_row_count()` | Count rows |
| `spark_utils` | `move_column_to_front_of_dataframe()` | Reorder columns |
| `spark_utils` | `repartition_and_cache()` | Optimize partitioning |
| `spark_utils` | `write_df_to_parquet()` / `read_parquet_to_df()` | Parquet I/O |
| `spark_utils` | `flatten_json_column_and_join_back_to_df()` | JSON flattening |
| `spark_utils` | `pivot_summary_table_for_bools()` | Boolean summaries |
| `spark_utils` | `pivot_summary_with_metrics()` | Categorical pivots |
| `spark_utils` | `atomic_write_with_staging()` | Safe file writes |
| `spark_utils` | `validate_geometry()` | Sedona geometry validation |
| `spark_utils` | `reproject_geom_columns()` | CRS reprojection |
| `spark_utils` | `compute_walkability()` | Distance grading |

### Dual-Mode Pattern

```python
# Pandas (always runs)
result = pdf.groupby('col').mean()

# Spark (only when available)
if SPARK_AVAILABLE and spark:
    sdf_result = sdf.groupBy('col').mean()
```

This pattern is used consistently across NB04, NB07, and this notebook.